# Earnalism retouch job `selfish-giant-retouch-film-01-r9`

**Prepared, not run.** Input zip SHA256 `28a78c985b71c5f37684a93a6d4e267687492cd3dc511ac05fd0e3cac4b6a30e`. Set `RUN = True` in the first cell, then run all. With the R2 exchange (`exchange.json` in `/content/drive/MyDrive/earnalism-audiovisual/the-selfish-giant/selfish-giant-retouch-film-01-r9/`) the notebook downloads `inputs.zip` itself and uploads the outputs when it finishes; without it, copy `inputs.zip` into that Drive folder first. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 2311 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'selfish-giant-retouch-film-01-r9'
TITLE = 'the-selfish-giant'
KIND = 'retouch'
INPUT_ZIP_SHA256 = '28a78c985b71c5f37684a93a6d4e267687492cd3dc511ac05fd0e3cac4b6a30e'
MODEL_REPO = 'diffusers/stable-diffusion-xl-1.0-inpainting-0.1'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 30
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/the-selfish-giant/selfish-giant-retouch-film-01-r9'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 2311
MANIFEST_RESERVE_S = 60
PACKAGES = ('diffusers', 'transformers', 'accelerate', 'safetensors', 'torch', 'huggingface_hub')
EXCHANGE_SCHEMA = 'earnalism.colab-exchange.v1'
MASK_REPO = 'CIDAS/clipseg-rd64-refined'
LICENCE_ALLOWLIST = ('apache-2.0', 'mit', 'openrail++')
GUIDANCE = 7.5
MASK = {'threshold': 0.4, 'dilate_px': 6, 'feather_px': 4}


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import shutil, subprocess
# A GPU first: a CPU-only runtime would install, download and then fail with a confusing CUDA error.
_smi = shutil.which("nvidia-smi")
if not _smi or subprocess.run([_smi, "-L"], capture_output=True).returncode != 0:
    raise RuntimeError("No GPU attached: Runtime -> Disconnect and delete runtime, then Runtime -> "
                       "Change runtime type -> A100 GPU (or L4), Save, Connect, and Run all again")
print(subprocess.run([_smi, "-L"], capture_output=True, text=True).stdout.strip())


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
_token, _secret_problem = None, None
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN") or None
except Exception as _e:  # no secret, notebook access not granted, or the prompt timed out
    _secret_problem = type(_e).__name__
    _token = os.environ.get("HF_TOKEN") or None
if _token:
    os.environ["HF_TOKEN"] = _token
from huggingface_hub import HfApi
try:
    HfApi().auth_check(MODEL_REPO, token=_token)
except Exception as _e:
    _fix = ("accept the model's terms at https://huggingface.co/" + MODEL_REPO + " with your Hugging "
            "Face account, put a Read token from https://huggingface.co/settings/tokens in the Colab "
            "secret HF_TOKEN (key icon in the left sidebar, Notebook access on), then Runtime > "
            "Restart session and Run all")
    if _secret_problem:
        _fix += " (the HF_TOKEN secret could not be read: %s; grant access if Colab asks)" % _secret_problem
    raise RuntimeError("No access to " + MODEL_REPO + " (" + type(_e).__name__ + "): " + _fix) from None
print("model access OK:", MODEL_REPO, "(token from HF_TOKEN)" if _token else "(public)")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
from huggingface_hub import HfApi
LICENCES = {}
for _repo in (MODEL_REPO, MASK_REPO):
    try:
        HfApi().auth_check(_repo, token=os.environ.get("HF_TOKEN"))
        _info = HfApi().model_info(_repo, token=os.environ.get("HF_TOKEN"))
    except Exception as _e:
        raise RuntimeError("No access to " + _repo + " (" + type(_e).__name__ + "); nothing ran") from None
    _card = getattr(_info, "card_data", None) or getattr(_info, "cardData", None) or {}
    _lic = _card.get("license") if hasattr(_card, "get") else getattr(_card, "license", None)
    if not _lic:
        _lic = next((t.split(":", 1)[1] for t in (getattr(_info, "tags", None) or [])
                     if t.startswith("license:")), None)
    _lic = str(_lic).strip().lower() if _lic else None
    print("model card licence:", _repo, "->", _lic, "(allowlist: %s)" % ", ".join(LICENCE_ALLOWLIST))
    if _lic not in LICENCE_ALLOWLIST:
        raise RuntimeError("model card licence of " + _repo + " is " + repr(_lic) + ", not one of "
                           + ", ".join(LICENCE_ALLOWLIST) + ": this job never runs it; tell Claude")
    LICENCES[_repo] = _lic


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
# R2 exchange (references/colab-exchange.md): exchange.json in this Drive folder carries two
# short-lived single-object URLs (GET inputs.zip, PUT outputs.tar). They are capabilities: never
# print them, never copy them elsewhere.
EXCHANGE = None
if (OUT_DIR / "exchange.json").is_file():
    import time, urllib.request
    from datetime import datetime
    EXCHANGE = json.loads((OUT_DIR / "exchange.json").read_text())
    if EXCHANGE.get("schema") != EXCHANGE_SCHEMA or EXCHANGE.get("job_id") != JOB_ID:
        raise RuntimeError("exchange.json belongs to a different job; ask Claude for this job's exchange.json")
    _left = datetime.fromisoformat(EXCHANGE["expires_at"].replace("Z", "+00:00")).timestamp() - time.time()
    if _left < DEADLINE_SECONDS + MANIFEST_RESERVE_S + 300:
        raise RuntimeError("the exchange URLs expire before this job's hard stop (%.0f s left); "
                           "ask Claude to re-sign them before running" % _left)
    if not INPUT_ZIP.is_file():
        with urllib.request.urlopen(EXCHANGE["inputs"]["url"], timeout=600) as _r:
            if not 200 <= _r.status < 300:
                raise RuntimeError("inputs.zip download failed: HTTP %d" % _r.status)
            _data = _r.read()
        _sha = hashlib.sha256(_data).hexdigest()
        if _sha != INPUT_ZIP_SHA256 or _sha != EXCHANGE["inputs"]["sha256"]:
            raise RuntimeError("downloaded inputs.zip sha256 mismatch: " + _sha + " (not kept)")
        INPUT_ZIP.write_bytes(_data)
        print("inputs.zip downloaded from the exchange:", len(_data), "bytes")
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Retouch: redraw only small masked regions of a chosen still (SDXL inpainting 0.1, CLIPSeg masks).
# Both exact revisions are resolved now and recorded; the licence preflight already ran.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os, gc, hashlib, io, math, subprocess, sys, zipfile
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers",
                "accelerate", "safetensors"], check=True)


# --- helpers (pure PIL; unit-tested by the skill)
import math
from PIL import Image, ImageChops, ImageFilter


def pad8(img):
    """Pad to multiples of 8 (SDXL's latent grid); the original pixels stay at (0, 0)."""
    w, h = img.size
    W, H = int(math.ceil(w / 8) * 8), int(math.ceil(h / 8) * 8)
    if (W, H) == (w, h):
        return img.copy()
    canvas = img.resize((W, H), Image.BICUBIC)  # the border strip continues the picture
    canvas.paste(img, (0, 0))
    return canvas


def union_mask(soft, threshold, dilate_px, feather_px):
    """Soft per-region masks (L, 0-255) -> thresholded, dilated, unioned, feathered outward."""
    cut = int(round(threshold * 255))
    union = None
    for m in soft:
        hard = m.point(lambda v: 255 if v >= cut else 0)
        union = hard if union is None else ImageChops.lighter(union, hard)
    if dilate_px > 0:
        union = union.filter(ImageFilter.MaxFilter(2 * dilate_px + 1))
    if feather_px > 0:  # the dilated region stays fully redrawn; only its edge blends outward
        union = ImageChops.lighter(union.filter(ImageFilter.GaussianBlur(feather_px)), union)
    return union


def composite(source, generated, mask):
    """Back at the source size: the generated pixels only where the mask is, the source elsewhere."""
    gen = generated.crop((0, 0) + source.size).convert(source.mode)
    return Image.composite(gen, source, mask)
# --- end helpers


import torch
from huggingface_hub import HfApi
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

revision = HfApi().model_info(MODEL_REPO).sha
mask_revision = HfApi().model_info(MASK_REPO).sha
gpu = torch.cuda.get_device_properties(0)
print("GPU", gpu.name, "%.1f GiB" % (gpu.total_memory / 2**30))

# Sources: every PNG is checked against the sha256 the production record holds.
SOURCES = {}
with zipfile.ZipFile(INPUT_ZIP) as z:
    for item in JOB["items"]:
        if item["source"] not in SOURCES:
            data = z.read(item["source"])
            if hashlib.sha256(data).hexdigest() != item["source_sha256"]:
                raise RuntimeError("source still sha256 mismatch: " + item["source"])
            SOURCES[item["source"]] = Image.open(io.BytesIO(data)).convert("RGB")

# Phase 1: masks (CLIPSeg alone on the GPU), one per still, written as <key>--mask.png for review.
processor = CLIPSegProcessor.from_pretrained(MASK_REPO, revision=mask_revision)
segmenter = CLIPSegForImageSegmentation.from_pretrained(MASK_REPO, revision=mask_revision).to("cuda")
MASKS, empty = {}, []
with torch.no_grad():
    for item in JOB["items"]:
        if item["base"] in MASKS:
            continue
        img = SOURCES[item["source"]]
        finds = [r["find"] for r in item["regions"]]
        inputs = processor(text=finds, images=[img] * len(finds), padding="max_length",
                           return_tensors="pt").to("cuda")
        logits = segmenter(**inputs).logits
        if logits.dim() == 2:
            logits = logits.unsqueeze(0)
        soft = []
        for find, prob in zip(finds, torch.sigmoid(logits).float().cpu()):
            m = Image.fromarray((prob.numpy() * 255).astype("uint8"), "L").resize(img.size, Image.BILINEAR)
            if max(m.getdata()) < int(round(MASK["threshold"] * 255)):
                empty.append(item["base"] + ": " + find)
            soft.append(m)
        mask = union_mask(soft, MASK["threshold"], MASK["dilate_px"], MASK["feather_px"])
        mask.save(OUT_DIR / (item["base"] + "--mask.png"))
        MASKS[item["base"]] = mask
print("masks:", len(MASKS), "; regions found nothing:", empty or "none")
del segmenter, processor
gc.collect()
torch.cuda.empty_cache()

# Phase 2: SDXL inpainting at the source resolution (padded to multiples of 8, cropped back).
from diffusers import StableDiffusionXLInpaintPipeline
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    MODEL_REPO, revision=revision, torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.set_progress_bar_config(disable=True)
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "mask_repo": MASK_REPO,
              "mask_revision": mask_revision, "licences": LICENCES, "steps": STEPS,
              "guidance": GUIDANCE, "mask": MASK, "gpu": gpu.name, "precision": "fp16",
              "empty_regions": empty, "outside_mask": "source pixels, unchanged"}
EXPECTED = [i["key"] + ".png" for i in JOB["items"]] + [m + ".png" for m in JOB["masks"]]
last = 0.0  # seconds the previous candidate took; the next starts only if it fits (R25)
for item in JOB["items"]:
    if not may_start(last):
        break
    started = time.time()
    src, mask = SOURCES[item["source"]], MASKS[item["base"]]
    image, mask8 = pad8(src), pad8(mask)
    regions = "; ".join(r["prompt"] for r in item["regions"])
    k = item["prompt"].find("Earnalism house style")
    scene, style = (item["prompt"][:k].strip(), item["prompt"][k:]) if k >= 0 else (item["prompt"], "")
    gen = torch.Generator("cpu").manual_seed(item["seed"])
    with torch.no_grad():
        out = pipe(prompt=regions + ". " + scene, prompt_2=(style + " " + regions).strip(),
                   image=image, mask_image=mask8, width=image.size[0], height=image.size[1],
                   num_inference_steps=STEPS, strength=item["strength"], guidance_scale=GUIDANCE,
                   generator=gen).images[0]
    composite(src, out, mask).save(OUT_DIR / (item["key"] + ".png"))
    last = time.time() - started
    print("wrote", item["key"], "in %.1f s" % last)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if (p.is_file() and p.name not in ("manifest.json", "inputs.zip", "exchange.json")
            and p.suffix != ".ipynb"):  # Colab keeps the job notebook in this folder
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
if globals().get("EXCHANGE") is not None:  # R2 exchange: one uncompressed tar, PUT once
    import os, tarfile, tempfile, urllib.error, urllib.request
    _fd, _tar = tempfile.mkstemp(suffix=".tar")  # local disk, never the Drive job folder
    os.close(_fd)
    with tarfile.open(_tar, "w", format=tarfile.PAX_FORMAT) as _t:
        for _name in ["manifest.json"] + sorted(outputs):
            _t.add(str(OUT_DIR / _name), arcname=_name, recursive=False)
    _size = os.path.getsize(_tar)
    _headers = dict(EXCHANGE["outputs"]["headers"], **{"Content-Length": str(_size)})
    with open(_tar, "rb") as _body:
        _req = urllib.request.Request(EXCHANGE["outputs"]["url"], data=_body, method="PUT",
                                      headers=_headers)
        try:
            with urllib.request.urlopen(_req, timeout=1800) as _r:
                _status = _r.status
        except urllib.error.HTTPError as _e:
            _status = _e.code
    os.unlink(_tar)
    print("outputs.tar upload: HTTP", _status, "(%d bytes)" % _size)
    if not 200 <= _status < 300:
        raise RuntimeError("outputs.tar upload failed: HTTP %d; the outputs stay in Drive, tell Claude"
                           % _status)
    print("outputs uploaded to the exchange; tell Claude the CU Colab shows")
